<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

Add text here

In [ ]:
# add code here

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

Add text here

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Add prompts here.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.